# Population Response Coding Model: Parameter-Fitting Walkthrough

This notebook gives a guided walkthrough of the main features 
of the parameter-fitting script: `ParamFits.py` 

The script fits population-code model parameters to behavioural data by repeatedly simulating model output, comparing it with target data, and using differential evolution to minimise the fit error.

## 1. What the script is trying to do

At a high level, the script asks:

> Which model parameters make the simulated behavioural scores best-match the empirical behavioural scores?

Fit oblique effect index `(OEI)` values to any combination of:

1. group-average behavioural data;
2. individual-participant behavioural data;
3. one task at a time, or both tasks jointly;
4. different decoder rules for the two tasks, contrast detection (`sum`, `mag`, `peak`) and orientation discrimination (`WTA`, `PV`, `ML`).

The optimisation itself is handled using `scipy.optimize.differential_evolution`;
- Searches over bounded parameter ranges for the parameter values associated with the lowest error.
- Key search parameters:  
&nbsp;&nbsp;&nbsp;&nbsp;- popsize: controls the number of candidate solutions explored in each generation  
&nbsp;&nbsp;&nbsp;&nbsp;- maxiter: defines the maximum number of generations before the search stops  
- In each generation, new solutions are created by combining and modifying existing candidates, and retained only when they improve on the current population (i.e. produce lower error fits).

## 2. Main imports and dependencies

The script combines standard scientific Python tools with custom project modules. The custom modules are where most of the model-specific simulation logic lives.

In [25]:
import copy
import itertools as itls
from multiprocessing import Pool
import numpy as np
from numpy import asarray as arr
import os
import pandas as pd
from scipy.optimize import differential_evolution

# Custom analysis utilities
from mytools.analysis import get_ave_data
from mytools.dframe_tools import custom_sort

# Model-specific components
from components.model_simulation import ModelSimulator

## 3. The central optimisation loop

The optimisation pipeline is organised around three main functions/classes:

| Component | Role                                                                   |
|---|------------------------------------------------------------------------|
| `run_de()` | Runs one differential-evolution search for a given seed.         |
| `run_objective_fn()` | Evaluates one candidate parameter vector and returns its error.        |
| `ObjectiveFunction` | Runs the model simulation and computes the error from behavioural data. |


In [26]:
# Global variable updated on each call to run_objective_fn() to save results from each candidate parameter set
de_simulations = {
    '1. Initial': [],
    '2. Rerun': []
}  

# Call differential evolution optimiser
def run_de(objH, param_names, bounds, popsize, maxiter, label, seed=None):
    
    """Run differential-evolution search."""
    de_fit = differential_evolution(
        lambda x: run_objective_fn(
            x,
            param_names,
            objH,
            n_eval=1, 
            n_level_reps=50,  # Number of repeats for each point along the psychometric function
            label=label,
            seed=seed
        ),
        bounds=bounds,  # Search within bounded parameter range
        
        popsize=popsize,  # Controls search diversity by setting how many candidate solutions are used per parameter.
        maxiter=maxiter,  # Maximum number of generations/iterations before stopping.
        
        polish=False,
        seed=seed
    )
    
    # --- PRINT RESULTS --- #
    # Anisotropy values back to linear scale for fractional free parameters - for print only
    _PRINT = True
    if _PRINT:
        print_fit = {_k: _v for _k, _v in zip(param_names, de_fit.x)}
        for k, v in print_fit.items():
            print_fit[k] = np.exp(v) if k.endswith('_fraction') else v
    
        print(f"\nBest fitting parameters: {print_fit}")
        print(f"\nBest fitting objective (i.e. error): {de_fit.fun}")
        print(f"\n{'=' * 60}\n")
    
    return de_fit


### Why several differential-evolution runs?

The script runs differential evolution multiple times using different random seeds. This reduces dependence on a single stochastic search path. After these runs, all candidate fits are pooled, and the best-performing subset is retained for more careful re-evaluation.

## 4. The objective function

`run_objective_fn()` is the bridge between the optimiser and the model. It receives one proposed parameter vector, runs the model, extracts the resulting error, and returns that error to the optimiser.

A useful detail is that parameters ending in `_fraction` are fitted on a log scale. This makes ratios such as `1/4` and `4` symmetric around zero after transformation.

In [27]:
def run_objective_fn(param_values, param_names, objH, n_eval, 
                     label='',
                     seed=None,
                     n_level_reps=None,
                     _return_output=False
                     ):
    
    """Evaluate one candidate parameter vector."""

    # Change simulation repetitions if requested.
    if n_level_reps is not None:
        objH.config['n_level_reps'] = n_level_reps

    # Run the model and compute the fit error.
    model_output, var_params = objH.run(param_values, param_names, n_eval)
    
    model_output['seed'] = seed
    
    total_error = model_output['total_error'].iloc[0]

    # Save full output if this candidate should be retained for later analysis.
    if label:
        global de_simulations
        
        if de_simulations.get(label) is None:
            de_simulations[label] = []
            
        de_simulations[label].append(model_output)
    
    # Outside of optimisation routine
    if _return_output:
        return model_output
    
    # Inside optimisation routine
    else:
        return total_error

## 5. How model error is computed

The `ObjectiveFunction` class runs simulations for each task/decoder pair. For each pair, it:

1. simulates model performance;
2. extracts the requested measure, such as `OEI` or `thresholds`;
3. retrieves the matching behavioural target values;
4. computes mean squared error;
5. stores the error alongside the model output.

This is the part of the script that decides whether a parameter set is a good or poor fit.

In [28]:
class ObjectiveFunction:
    """Computes error between model output and target data."""
    def __init__(self, simulator, tasknames, decoders, measure, config, error_type='lin', task_weights=None):
        
        self.simulator = simulator
        self.tasknames = [tasknames] if not isinstance(tasknames, (list, tuple)) else tasknames
        self.decoders = [decoders] if not isinstance(decoders, (list, tuple)) else decoders

        # self.task_names = task_names if isinstance(task_names, list) else [task_names]
        self.measure = measure
        self.config = config
        self.error_type = error_type
        self.task_weights = task_weights or {_t: 1.0 for _t in self.tasknames}
        
    def run(self, param_values, param_names, n_eval=1):
        """
        Compute error between model predictions and target data.
        This is the central scoring routine used by the optimiser.
        """
        var_params = {name: value for name, value in zip(param_names, param_values)}
        
        total_error = 0
        all_model_output = []

        # Looping through different tasks and decoders is set up to allow fitting multiple models at once
        # This is not done in this example, rather when fitting both tasks w/ shared model in `ParamFits.py`
        
        # Conversion of fractional free parameters to desired scale is handled by ModelSimulator
        
        for i_task, i_decoder in zip(self.tasknames, self.decoders):
            results = self.simulator.simulate(
                i_task,
                i_decoder,
                self.measure,
                var_params,
                self.config,
                n_eval
            )

            model_output = pd.DataFrame(results)
            
            # Record which parameters were free in this model variant.
            for fp_idx, free_param in enumerate(param_names):
                model_output[f"freeParam_{fp_idx}"] = free_param

            # Convert model output into the same format as behavioural target data
            model_scores = arr(model_output[self.measure])
            
            # Behavioural target values for the current task and measure.
            target_scores = np.hstack(self.config[self.measure][i_task])
            
             # Compute the fit error.
            task_error = np.mean((model_scores - target_scores) ** 2)
                       
            _PRINT = False
            if _PRINT:
                print(f"\n{i_task} & {i_decoder}")
                print(f"  Target: {np.round(target_scores, 3)}")
                print(f"  Model:  {np.round(model_scores, 3)}")
            
            model_output['fit_error'] = task_error
            all_model_output.append(model_output)
        
            total_error += task_error
        
        # Store the actual objective value used for optimisation.
        all_model_output = pd.concat(all_model_output)
        all_model_output['total_error'] = total_error
        
        # print(f"\nTotal Error: {total_error:.4f}")
    
        return all_model_output, var_params

## 6. Full optimisation strategy

The `optimize_parameters()` function manages the full fitting workflow.

The key idea is to use a relatively fast, noisy optimisation stage first, then re-run the best candidate fits with more repetitions to obtain more stable estimates.

The stages are:

1. define free-parameter names and bounds;
2. run differential evolution several times;
3. pool all fits across runs;
4. keep the top 5% of candidate fits;
5. re-run the top fits with more repetitions;
6. remove unstable outliers;
7. take the median parameter values from the retained fits;
8. run and save the final model fit.

In [29]:
def optimize_parameters(objH, free_params, config, save_dir=''):
    
    # ------------------------
    # 1. define free-parameter names and bounds;
    # ------------------------

    # These lists keep names, initial values, and bounds aligned.
    param_names = list(free_params.keys())
    initial_values = [free_params[k]['x0'] for k in param_names]  # unused in diff_evol
    bounds = [free_params[k]['bounds'] for k in param_names]

    # Build a filename that records which parameters were free in this fit.
    params_str = ''.join(f"--{_i}" for _i in free_params.keys()).lstrip('-')
    filename = f"{objH.tasknames}_{objH.decoders}--{params_str}"

    # ------------------------
    # 2. run differential evolution several times;
    # ------------------------
    print(f"\n>>> 2. run differential evolution several times;")

    # define random seeding
    seeds = np.random.choice(1000, size=config['N_runs'], replace=False)
    
    # Prepare list of lists containing arguments for each DE search.
    args_list = [(objH, param_names, bounds, config['popsize'], config['max_iter'], '1. Initial', seed
                  ) for seed in seeds]
    
    # Main script uses multiprocessing, but easier without for example to allow debugging
    _use_multiprocessing = False
    if _use_multiprocessing:
        with Pool(processes=min(config['N_runs'], 3)) as pool:
            all_de_results = pool.starmap(run_de, args_list)
    else:
        all_de_results = [
            run_de(*args)
            for args in args_list
        ]

    # ------------------------
    # 3. pool all fits across runs;
    # ------------------------    
    global de_simulations
    fits_initial = pd.concat(de_simulations['1. Initial'])
    
    print(f"\n3. Total number of model iterations within optimisation routine: {len(fits_initial)}")
    
    # ------------------
    # 4. keep the top 5% of candidate fits;
    # ------------------

    # Keep only the strongest candidate fits for the more expensive re-evaluation stage.
    n_candidate_sets = len(fits_initial.drop_duplicates(subset=param_names))  # n_unique candidates
    quantile_cutoff = 0.05 if n_candidate_sets > 50 else 0.2

    top_fits = fits_initial[fits_initial['total_error'] <= fits_initial['total_error'].quantile(quantile_cutoff)]

    # ---------------------
    # 5. re-run the top fits with more repetitions;
    # ---------------------
    print(f"\n>>> 5. re-run the top fits with more repetitions")
    # Re-evaluate each unique candidate parameter set. 
    
    # Use info from a single SF condition to prevent repeated simulations (shared fits across SF)
    for idx, row in top_fits[top_fits['cond'] == top_fits['cond'].tolist()[0]].iterrows():

        # Extract parameter values from the selected row.
        vals = row[param_names].tolist()
        
        # Convert anisotropy fraction to the log scale expected by objective function.
        for _idx, _lbl in enumerate(param_names):
            vals[_idx] = np.log(vals[_idx]) if _lbl.endswith('_fraction') else vals[_idx]
                
        # Re-simulate selected candidates w/ more reps for a more stable error estimate
        # Updates global variables
        run_objective_fn(
            vals, 
            param_names, 
            objH, 
            n_eval=1, 
            n_level_reps=500, 
            label='2. Rerun',
        )

    # ---------------------
    # 6. remove unstable outliers;
    # ---------------------
    
    # Remove fits whose re-evaluated error is much larger than the typical error
    
    # Replace the original top fits with their re-evaluated outputs.
    top_fits = pd.concat(de_simulations['2. Rerun'])
    display(top_fits)
        
    # Remove fits with error values > 2x the median error for that condition
    top_fits = top_fits[
        top_fits['total_error'] <= top_fits.groupby(
            ['task', 'decoder', 'cond']
        )['total_error'].transform('median') * 2
        ].reset_index(drop=True)  # ungrouped, clean index

    # -----------------------
    # 7. take the median parameter values from the retained fits;
    # -----------------------

    # Use median of re-evaluated top fits as final parameter estimate. 
    final_param_vals = top_fits[param_names].median().to_list()
    
    print(f"\nFinal parameter values: {np.round(final_param_vals, 3)}"
          f"\nfor free parameters: {free_params.keys()}")
    
    # Fraction parameters converted back to log scale.
    for idx, param_name in enumerate(param_names):
        if param_name.endswith('_fraction'):
            final_param_vals[idx] = np.log(final_param_vals[idx])

    # ----------------------
    # 8. run and save the final model fit.
    # ----------------------

    # Run a high-precision final simulation and save the resulting model output.
    final_output = run_objective_fn(
        final_param_vals, param_names, objH,
        n_eval=1,
        n_level_reps=100,
        _return_output=True
    )
    final_output.to_csv(os.path.join(save_dir, f'optFit-{filename}.csv'), index=False)

    # ----------
    # Calculate summary data and save
    # ----------
    
    # Summarise the distribution of selected top fits by task, decoder, and condition.
    numeric_cols = top_fits.select_dtypes(include="number").columns
    measure_order = ['median', 'q25', 'q75', 'mean', 'std', 'sem']

    top_summary = get_ave_data(top_fits, ['task', 'decoder', 'cond'], numeric_cols)
    top_summary = custom_sort(
        top_summary,
        ['measure', 'task', 'decoder', 'cond'],
        [measure_order, ['con_det', 'ori_dis'], [], ['low', 'mid', 'high']]
    )

    # Save the robust top-fit table.
    top_fits.round(5).to_csv(os.path.join(save_dir, f'topFits-{filename}.csv'))

    # Save summary statistics for the selected top fits.
    top_summary.round(5).to_csv(os.path.join(save_dir, f'summaryTopFits_{filename}.csv'))

    return None

## 7. Behavioural targets and simulation settings

The `config` dictionary contains both simulation settings and behavioural target data. This is important because the model output is scored directly against these target values.

For example, when `MEASURE = 'OEI'`, the optimiser compares simulated OEI values against the entries in `config['OEI']`.

In [30]:
MEASURE = 'OEI'
TOTAL_N = 60  # 540 in formal simulations - 60 used here for speed

config = {
    'run_type': 'loop',
    'design': 'mocs',
    
    'std_oris': [0, 45],
    'contrasts': [100],
    'n_level_reps': 10,
    
    'step_size': 0.02,  # Tiling of orientation space

    # Behavioural targets
    'OEI': {
        'con_det': [1.06, 1.39, 1.71],
        'ori_dis': [2.45, 3.25, 3.36]
    },
    
    # Differential Evolution Settings (smaller values than in formal modelling)
    'N_runs': 3,  # Number of repeats of initial DE routine
    'popsize': 3,  # Population size of DE
    'max_iter': 3,  # Max iter of DE
}

## 8. Spatial-frequency conditions

The script uses three spatial-frequency conditions: `low`, `mid`, and `high`. Each condition has a different tuning width parameter.

These condition labels allow the output to be grouped and summarised by condition after fitting.

In [31]:
cond_labels = ['low', 'mid', 'high']
sf_params = {'fwhm': [56.2, 30.0, 22.0]}  # Bandwidth here is defined as full-width half max

## 9. Free parameters and model variants

The script does not fit just one model. It generates every non-empty combination of four anisotropy parameters:

- `n_fraction`
- `fwhm_fraction`
- `noise_scale_fraction`
- `rmax_fraction`

Each combination of anisotropies can be treated as a separate model variant. This allows us to compare whether different sources of anisotropy are sufficient to explain the behavioural data.

In [32]:
allFreeParams = [
    'n_fraction',
    'fwhm_fraction',
    'noise_scale_fraction',
    'rmax_fraction'
]

freeParamCombos = []
for r in range(1, len(allFreeParams) + 1):
    freeParamCombos.extend(itls.combinations(allFreeParams, r))

print(f"\nParameter Combinations")
display(freeParamCombos[:])


Parameter Combinations


[('n_fraction',),
 ('fwhm_fraction',),
 ('noise_scale_fraction',),
 ('rmax_fraction',),
 ('n_fraction', 'fwhm_fraction'),
 ('n_fraction', 'noise_scale_fraction'),
 ('n_fraction', 'rmax_fraction'),
 ('fwhm_fraction', 'noise_scale_fraction'),
 ('fwhm_fraction', 'rmax_fraction'),
 ('noise_scale_fraction', 'rmax_fraction'),
 ('n_fraction', 'fwhm_fraction', 'noise_scale_fraction'),
 ('n_fraction', 'fwhm_fraction', 'rmax_fraction'),
 ('n_fraction', 'noise_scale_fraction', 'rmax_fraction'),
 ('fwhm_fraction', 'noise_scale_fraction', 'rmax_fraction'),
 ('n_fraction', 'fwhm_fraction', 'noise_scale_fraction', 'rmax_fraction')]

## 10. Log-transforming fraction parameters

Fraction parameters are ratios, for example cardinal:oblique ratios. The script log-transforms these parameters before fitting.

This has two advantages:

1. ratios below and above 1 become symmetric around zero;
2. the optimiser searches multiplicative changes more naturally.

For example, `1/4` and `4` become equally far from zero on the log scale.

In [33]:
free_params_settings = {
    'n_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'fwhm_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'rmax_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'noise_scale_fraction': {'x0': 1, 'bounds': (1 / 4, 4)}
}

# log-transform bounds and initial value.
example_settings = copy.deepcopy(free_params_settings)

for param, fit_constraints in example_settings.items():
    
    if not param.endswith(f"_fraction"):
        continue

    for k in fit_constraints:
        example_settings[param][k] = np.log(example_settings[param][k])
        
display('Original settings:', free_params_settings)
display('Transformed:', example_settings)

'Original settings:'

{'n_fraction': {'x0': 1, 'bounds': (0.25, 4)},
 'fwhm_fraction': {'x0': 1, 'bounds': (0.25, 4)},
 'rmax_fraction': {'x0': 1, 'bounds': (0.25, 4)},
 'noise_scale_fraction': {'x0': 1, 'bounds': (0.25, 4)}}

'Transformed:'

{'n_fraction': {'x0': np.float64(0.0), 'bounds': array([-1.386,  1.386])},
 'fwhm_fraction': {'x0': np.float64(0.0), 'bounds': array([-1.386,  1.386])},
 'rmax_fraction': {'x0': np.float64(0.0), 'bounds': array([-1.386,  1.386])},
 'noise_scale_fraction': {'x0': np.float64(0.0),
  'bounds': array([-1.386,  1.386])}}

# 11. Default model parameters
### Defines the default parameter values of the model.
### Overwritten by free_params in case of conflict

In [34]:
# --------------------------------------------------------------------
# Default model parameters
# --------------------------------------------------------------------

# These defaults define the baseline model. Any free parameters selected
# above will be overwritten during optimisation.
default_params = {
    # PARAMS
    'total_n': TOTAL_N, 'fwhm': 50, 'rmax': 60, 'noise_scale': 1,
    'spont': 1, 'crf_exponent': 2, 'crf_c50': 24,
    'pooling_q': 7, 'param_distrib': 'cos', 'pref_distrib': 'cos',
    # ANISOTROPIES
    'n_fraction': 1, 'fwhm_fraction': 1, 'rmax_fraction': 1, 'noise_scale_fraction': 1,
    'crf_exponent_fraction': 1, 'crf_c50_fraction': 1, 'spont_fraction': 1,
}

## 12. Average-data fitting branch

When `FIT_AVERAGE = True`, the script fits the model to the group-average behavioural values in `config`.

If `FIT_BOTH_TASKS = False`, each task and decoder is fitted separately. 

If `FIT_BOTH_TASKS = True`, the script fits both tasks together for each pair of decoders.

In [35]:
FIT_AVERAGE = True
FIT_BOTH_TASKS = False

# 13. Run parameter fitting routine - Fit to AVERAGE DATA

In [36]:
# Used to convert param names into short strings for saving
param_str_dict = {'n_fraction': 'Nf', 'fwhm_fraction': 'Hf', 'noise_scale_fraction': 'NZf', 'rmax_fraction': 'Rf'}
    
# Select only the free parameters for the current model variant.
for FREE_PARAMS_CHOICE in freeParamCombos:
    
    # Create a compact label for the current set of free parameters.
    param_id_str = ''.join(f"{param_str_dict.get(p, p)}_" for p in FREE_PARAMS_CHOICE).rstrip('_')

    # This title is used to name the output folder for the current model variant.
    TITLE = f"n{TOTAL_N}_{MEASURE}__{param_id_str}"
    
    # Select only the free parameters for the current model variant.
    free_params = {}
    for choice in FREE_PARAMS_CHOICE:
        # deepcopy() is crucial to keep values independent
        free_params[choice] = copy.deepcopy(free_params_settings[choice])  
    
    # In this example code I chose to only use the four parameter model
    if len(FREE_PARAMS_CHOICE) != 4:
        continue
    
    # Optimise fraction parameters on a log scale (reciprocal ratios symmetric about midpoint)
    for param, fit_constraints in free_params.items():
        if param.endswith(f"_fraction"):
            for k in fit_constraints:
                free_params[param][k] = np.log(free_params[param][k])
            
    # --------------------------------------------------------------------
    # Paths - Each model variant receives its own output directory
    # --------------------------------------------------------------------
    og_dir = os.getcwd()
    fits_dir = os.path.join(og_dir, 'test', TITLE)

    if not os.path.exists(fits_dir):
        os.makedirs(fits_dir)
    else:
        print(f"\nPath already exists for {TITLE}. Overwrite files with shared filenames")
    
    # --------------------------------------------------------------------
    # Simulation - Contrast Detection with unweighted (UW) pooling method
    # --------------------------------------------------------------------
    print("\n=======\nCONTRAST DETECTION\n=======")
    
    # Create simulator for the current model variant.
    simulator = ModelSimulator(default_params, cond_labels, sf_params)

    # Create objective function for the current task and decoder (one of ['sum', 'mag', 'peak'])
    objHandler = ObjectiveFunction(
        simulator=simulator, 
        tasknames='con_det', 
        decoders='sum', 
        measure=MEASURE,
        config=config, 
    )
    
    # Run the optimisation pipeline for this task/decoder pair.
    opt_fit_CONDET = optimize_parameters(objHandler, free_params, config, save_dir=fits_dir)
    
    # --------------------------------------------------------------------
    # Simulation - Orientation Discrimination with winner-take-all (WTA) orientation decoder
    # --------------------------------------------------------------------
    print("\n=======\nORIENTATION DISCRIMINATION\n=======")
    # Create objective function for the current task and decoder (one of ['ML', 'PV', 'WTA'])
    objHandler = ObjectiveFunction(
        simulator=simulator, 
        tasknames='ori_dis', 
        decoders='WTA', 
        measure=MEASURE,
        config=config, 
    )

    # Run the optimisation pipeline for this task/decoder pair.
    opt_fit_ORIDIS = optimize_parameters(objHandler, free_params, config, save_dir=fits_dir)
    


Path already exists for n60_OEI__Nf_Hf_NZf_Rf. Overwrite files with shared filenames

CONTRAST DETECTION

>>> 2. run differential evolution several times;

Best fitting parameters: {'n_fraction': np.float64(1.8789575130828255), 'fwhm_fraction': np.float64(0.8650392667531139), 'noise_scale_fraction': np.float64(3.1544139187930242), 'rmax_fraction': np.float64(1.0064714901489467)}

Best fitting objective (i.e. error): 0.007114252841203615



Best fitting parameters: {'n_fraction': np.float64(1.9879279730258999), 'fwhm_fraction': np.float64(0.8214023233038543), 'noise_scale_fraction': np.float64(0.32733698597720623), 'rmax_fraction': np.float64(0.5799500537060225)}

Best fitting objective (i.e. error): 0.011074816033997327





KeyboardInterrupt



## 14. Individual-participant fitting branch

In `ParamFits.py`, when `FIT_INDIVIDUAL = True`, the script loads participant-level OEI scores and fits model parameters for each participant.

Full code can be found in `ParamFits.py`


## 15. Outputs saved by the script

For each model variant, task, decoder, and fitting level, the script saves several CSV files:

| File prefix | Contents |
|---|---|
| `optFit-...` | Final model output using the median parameter estimate from robust top fits. |
| `topFits-...` | Re-evaluated top candidate fits after filtering. |
| `summaryTopFits_...` | Summary statistics for the retained top fits. |

These outputs are designed to support later comparison across model variants and decoders.

## 16. Key things to check when running the script

Before running a large batch of fits, it is worth checking:

1. whether the `TITLE` is accurately named;
2. whether `FIT_AVERAGE`, `FIT_INDIVIDUAL`, and `FIT_BOTH_TASKS` are set as intended;
3. whether the behavioural CSV files exist in `beh_data`;
4. whether `n_level_reps`, `popsize`, and `maxiter` are appropriate for the desired speed and precision;
5. whether multiprocessing behaves as expected on the machine being used.

For debugging, reduce `TOTAL_N`, `popsize`, `maxiter`, and `n_level_reps` before running the full fitting batch.

## 17. Model overview

A compact way to understand the `ParamFits.py` script is:

```text
Choose model variant
    ↓
Choose task and decoder
    ↓
Simulate model using candidate parameters
    ↓
Compare simulated scores with behavioural targets
    ↓
Use differential evolution to reduce error
    ↓
Re-run the best candidates for more precise estimates of error
    ↓
Save final parameters and model outputs
```

The central idea is that parameter fitting is driven entirely by the objective error. Each candidate parameter set is used to simulate model behaviour. If the simulated scores move closer to the behavioural targets, the parameter set is treated as a better fit.

## 18. Plots
Please see `fitGraphs.py` to plot the results from `ParamFits.py`